In [ ]:
#Cài thư viện
!pip -q install bertopic umap-learn hdbscan transformers torch sentencepiece underthesea openpyxl safetensors

In [ ]:
#Upload file
from google.colab import files

uploaded = files.upload()
FILE_PATH = next(iter(uploaded.keys()))

print("File:", FILE_PATH)

In [ ]:
#Lấy và đọc data
import pandas as pd
from pathlib import Path
import ast

TEXT_COL_INDEX = 4  # cot E
HAS_HEADER = True

def read_input_file(path):
    suffix = Path(path).suffix.lower()

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(
            path,
            header=0 if HAS_HEADER else None,
            engine="openpyxl" if suffix == ".xlsx" else None
        )

    for enc in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            df_try = pd.read_csv(path, header=0 if HAS_HEADER else None, encoding=enc)
            if df_try.shape[1] == 1:
                df_try = pd.read_csv(
                    path,
                    header=0 if HAS_HEADER else None,
                    encoding=enc,
                    sep=None,
                    engine="python"
                )
            return df_try
        except UnicodeDecodeError:
            continue

    raise ValueError("Khong doc duoc file. Kiem tra lai encoding hoac dinh dang file.")

df = read_input_file(FILE_PATH)

print("Shape:", df.shape)
print("Columns:", list(df.columns))

if df.shape[1] <= TEXT_COL_INDEX:
    raise ValueError(f"File chi co {df.shape[1]} cot, khong co cot E.")

texts = (
    df.iloc[:, TEXT_COL_INDEX]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

docs = texts.tolist()

work_df = df.copy()
work_df["bertopic_text"] = docs

print("Ten cot E:", df.columns[TEXT_COL_INDEX])
print("Tong so dong dua vao BERTopic:", len(docs))
print("So dong rong:", int((texts.str.len() == 0).sum()))

display(work_df[["bertopic_text"]].head())


def parse_bertopic_cell(text):
    try:
        phrases = ast.literal_eval(text)
        if isinstance(phrases, list):
            return " ".join(str(p).replace("_", " ") for p in phrases)
    except Exception:
        pass
    return text.replace("_", " ")

docs = [parse_bertopic_cell(d) for d in docs]

In [ ]:
#Embedding PHOBERT
import hashlib
import numpy as np
import torch

from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

PHOBERT_MODEL = "vinai/phobert-base-v2"
OUTPUT_DIR = Path("/content/outputs_bertopic")
CACHE_DIR = OUTPUT_DIR / "embedding_cache"
MAX_LENGTH = 128

def get_underthesea_tokenizer():
    try:
        import underthesea
        print("Dung underthesea.word_tokenize truoc PhoBERT.")
        return lambda text: underthesea.word_tokenize(str(text), format="text")
    except Exception:
        print("Khong dung duoc underthesea, se dung text goc.")
        return None

def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def build_phobert_embeddings(texts, cache_path, batch_size=0, max_length=128, force_recompute=False):
    if cache_path.exists() and not force_recompute:
        cached = np.load(cache_path)
        if cached.shape[0] == len(texts):
            print("Dung cache embedding:", cache_path)
            return cached

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    tokenize_fn = get_underthesea_tokenizer()
    if tokenize_fn:
        processed = [tokenize_fn(t) for t in tqdm(texts, desc="underthesea")]
    else:
        processed = list(texts)

    tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL, use_fast=False)
    model = AutoModel.from_pretrained(PHOBERT_MODEL).to(device)
    model.eval()

    if batch_size <= 0:
        batch_size = 64 if device == "cuda" else 16

    all_embeddings = []
    bs = batch_size
    i = 0

    progress = tqdm(total=len(processed), desc="PhoBERT embedding")

    while i < len(processed):
        batch = processed[i:i + bs]

        try:
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}

            with torch.inference_mode():
                out = model(**encoded)
                pooled = mean_pool(out.last_hidden_state, encoded["attention_mask"])

            all_embeddings.append(pooled.cpu().numpy().astype(np.float32))
            progress.update(len(batch))
            i += bs

        except RuntimeError as e:
            if "out of memory" in str(e).lower() and bs > 1:
                bs = max(1, bs // 2)
                torch.cuda.empty_cache()
                print("OOM, giam batch_size xuong:", bs)
                continue
            raise

    progress.close()

    embeddings = np.vstack(all_embeddings).astype(np.float32)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    embeddings = embeddings / norms

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache_path, embeddings)

    print("Da luu cache:", cache_path)
    return embeddings

cache_key = hashlib.sha256(
    ("\n".join(docs) + PHOBERT_MODEL + str(MAX_LENGTH)).encode("utf-8")
).hexdigest()[:16]

cache_path = CACHE_DIR / f"phobert_{cache_key}.npy"

embeddings = build_phobert_embeddings(
    docs,
    cache_path=cache_path,
    batch_size=0,
    max_length=MAX_LENGTH,
    force_recompute=False
)

print("Embeddings shape:", embeddings.shape)

In [ ]:
#Chạy BERTOPIC
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

MIN_CLUSTER_SIZE = 80
MIN_SAMPLES = 10
UMAP_NEIGHBORS = min(15, max(2, len(docs) - 1))
MIN_DF = 1 if len(docs) < 100 else 3

umap_model = UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
    init="random",
    low_memory=True
)

hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    ngram_range=(1, 3),
    min_df=MIN_DF,
    token_pattern=r"(?u)\b\w+\b"
)

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    top_n_words=15,
    verbose=True,
    min_topic_size=MIN_CLUSTER_SIZE
)

topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_outlier = topics.count(-1)

print("So topic:", n_topics)
print("So outlier topic -1:", n_outlier)

In [ ]:
#Xuất kết quả
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

work_df = work_df.copy()
work_df["topic_id"] = topics

topic_info = topic_model.get_topic_info()

keyword_map = {}

for _, row in topic_info.iterrows():
    tid = int(row["Topic"])
    topic_words = topic_model.get_topic(tid)

    if topic_words:
        keyword_map[tid] = ", ".join([w for w, score in topic_words[:10]])
    else:
        keyword_map[tid] = ""

work_df["topic_keywords"] = work_df["topic_id"].map(keyword_map).fillna("")

topic_summary = []

for tid in sorted(work_df["topic_id"].unique()):
    group = work_df[work_df["topic_id"] == tid]
    examples = " | ".join(group["bertopic_text"].dropna().astype(str).head(3).tolist())

    topic_summary.append({
        "topic_id": tid,
        "count": len(group),
        "pct_of_total": round(len(group) / len(work_df) * 100, 2),
        "keywords": keyword_map.get(tid, ""),
        "examples": examples,
        "ten_topic_thu_cong": "",
        "giu_lai": ""
    })

topics_df = pd.DataFrame(topic_summary)

assign_csv = OUTPUT_DIR / "topic_assignments.csv"
overview_xlsx = OUTPUT_DIR / "topics_overview.xlsx"

work_df.to_csv(assign_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(overview_xlsx, engine="openpyxl") as writer:
    topics_df.to_excel(writer, sheet_name="Topics", index=False)
    work_df.to_excel(writer, sheet_name="Assignments", index=False)
    topic_info.to_excel(writer, sheet_name="BERTopic_Info", index=False)

print("Da xuat:", assign_csv)
print("Da xuat:", overview_xlsx)

display(topics_df.head(30))

In [ ]:
#Tải file về
from google.colab import files

files.download(str(overview_xlsx))
files.download(str(assign_csv))